# Posttest 2 — Data Preprocessing (Dataset Zoo)

Nama: Dewi
Dataset: Zoo Dataset (UCI Machine Learning Repository) — https://archive.ics.uci.edu/dataset/111/zoo

Notebook ini lanjutan dari Posttest 1 (eksplorasi dataset), fokusnya di tahap pra-pemrosesan data: data cleaning, normalisasi, feature engineering, encoding, sama data splitting.

## 0. Setup Awal

Sebelum masuk ke tahap preprocessing, import dulu semua library yang dipakai, baca datasetnya, terus bikin kolom `class_name` (hasil mapping dari `class_type`) yang nanti dipakai di beberapa bagian bawah.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("zoo.csv")

class_map = {
    1: "Mammalia",
    2: "Aves",
    3: "Reptilia",
    4: "Pisces",
    5: "Amphibia",
    6: "Insecta",
    7: "Invertebrata"
}
df["class_name"] = df["class_type"].map(class_map)

df.head()


,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type,class_name
0,aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1,Mammalia
1,antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1,Mammalia
2,bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4,Pisces
3,bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1,Mammalia
4,boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1,Mammalia


## 1. Data Cleaning

### 1.1 Cek missing value

Cek dulu ada nilai kosong (NaN) apa nggak pakai `.isnull().sum()`.

In [3]:
print("Jumlah missing value per kolom:")
print(df.isnull().sum())
print()
print("Total missing value di seluruh dataset:", df.isnull().sum().sum())


Jumlah missing value per kolom:
animal_name    0
hair           0
feathers       0
eggs           0
milk           0
airborne       0
aquatic        0
predator       0
toothed        0
backbone       0
breathes       0
venomous       0
fins           0
legs           0
tail           0
domestic       0
catsize        0
class_type     0
class_name     0
dtype: int64

Total missing value di seluruh dataset: 0


Hasilnya nol semua, berarti dataset Zoo ini emang nggak ada missing value. Jadi nggak perlu imputasi atau hapus data di tahap ini.

### 1.2 Cek data duplikat

baris yang isinya sama persis di semua kolom, sama nama hewan (`animal_name`) yang muncul lebih dari sekali (idealnya kan satu nama cuma satu baris).

In [4]:
print("Jumlah baris duplikat penuh (semua kolom sama persis):", df.duplicated().sum())
print("Jumlah animal_name yang duplikat:", df["animal_name"].duplicated().sum())

dup_names = df[df["animal_name"].duplicated(keep=False)]
dup_names


Jumlah baris duplikat penuh (semua kolom sama persis): 0
Jumlah animal_name yang duplikat: 1


,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type,class_name
25,frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5,Amphibia
26,frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5,Amphibia


Ternyata nggak ada baris yang bener-bener identik, tapi ada nama yang sama yaitu `frog` muncul 2 kali (index 25 dan 26). Pas dicek lagi, dua baris ini beda di kolom `venomous` (0 sama 1) — jadi sebenernya bukan duplikat, cuma dua varian frog yang beda (yang satu nggak berbisa, satu lagi berbisa), kebetulan aja namanya sama.

Karena bukan duplikat asli, untuk bari ini tidak dihapus.

In [5]:
df.loc[(df["animal_name"]=="frog") & (df["venomous"]==1), "animal_name"] = "frog2"

print("Setelah diperbaiki, animal_name yang duplikat:", df["animal_name"].duplicated().sum())
df[df["animal_name"].isin(["frog", "frog2"])]


Setelah diperbaiki, animal_name yang duplikat: 0


,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type,class_name
25,frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5,Amphibia
26,frog2,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5,Amphibia


### 1.3 Cek outlier

Dari semua kolom angka, cuma `legs` yang bukan data biner, jadi cuma ini yang relevan buat dicek outlier-nya. Aku pakai metode IQR (Interquartile Range).

Aturannya: dianggap outlier kalau nilainya di bawah `Q1 - 1.5*IQR` atau di atas `Q3 + 1.5*IQR`.

In [7]:
Q1 = df["legs"].quantile(0.25)
Q3 = df["legs"].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Batas bawah = {batas_bawah}, Batas atas = {batas_atas}")

outlier_legs = df[(df["legs"] < batas_bawah) | (df["legs"] > batas_atas)]
outlier_legs[["animal_name", "legs", "class_name"]]


Q1 = 2.0, Q3 = 4.0, IQR = 2.0
Batas bawah = -1.0, Batas atas = 7.0


,animal_name,legs,class_name
53,octopus,8,Invertebrata
72,scorpion,8,Invertebrata


Ketemu 2 data yang secara hitungan statistik kayak outlier: octopus sama scorpion, sama-sama 8 kaki. Tapi kalau dipikir lagi, ini valid secara biologis kok — gurita emang punya 8 tentakel dan kalajengking emang 8 kaki, jadi bukan salah input, cuma emang variasi alami jumlah kaki antar hewan.

## 2. Normalisasi kolom angka

Hampir semua kolom angka di dataset ini udah dalam skala 0-1 (biner). Cuma `legs` yang skalanya beda (0-8). Biar semua kolom angka setara skalanya dan nggak ada yang "mendominasi", aku normalisasi `legs` pakai Min-Max Scaling supaya hasilnya juga di rentang 0-1.

Pakai `MinMaxScaler` dari `sklearn.preprocessing`.

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df["legs_scaled"] = scaler.fit_transform(df[["legs"]])

df[["animal_name", "legs", "legs_scaled"]].head(10)


,animal_name,legs,legs_scaled
0,aardvark,4,0.5
1,antelope,4,0.5
2,bass,0,0.0
3,bear,4,0.5
4,boar,4,0.5
5,buffalo,4,0.5
6,calf,4,0.5
7,carp,0,0.0
8,catfish,0,0.0
9,cavy,4,0.5


Sekarang `legs_scaled` udah di rentang 0-1, sama kayak kolom biner lainnya, jadi pas modeling nanti nggak ada kolom yang timpang skalanya.

## 3. Feature Engineering

Fitur 1 — `total_ciri_aktif` (numerik): total ciri biner yang bernilai 1 di tiap hewan (hair + feathers + eggs + milk + airborne + aquatic + predator + toothed + backbone + breathes + venomous + fins + tail + domestic + catsize). Semacam skor seberapa banyak ciri yang dimiliki hewan itu.

Fitur 2 — `habitat` (kategorikal, nggak ada urutan): dari kombinasi `aquatic` sama `airborne`, jadi 4 kategori: Air, Udara, Darat, atau Amfibi. Nominal karena nggak ada yang "lebih tinggi" dari yang lain.

Fitur 3 — `leg_group` (kategorikal, ada urutan/ordinal): dari `legs`, dikelompokin jadi Tanpa Kaki < Sedikit (2) < Sedang (4) < Banyak (6-8). Ordinal karena urutannya jelas dari sedikit ke banyak.

In [9]:
ciri_biner = ["hair","feathers","eggs","milk","airborne","aquatic","predator",
              "toothed","backbone","breathes","venomous","fins","tail","domestic","catsize"]

df["total_ciri_aktif"] = df[ciri_biner].sum(axis=1)

def tentukan_habitat(row):
    if row["aquatic"] == 1 and row["airborne"] == 0:
        return "Air"
    elif row["airborne"] == 1 and row["aquatic"] == 0:
        return "Udara"
    elif row["aquatic"] == 1 and row["airborne"] == 1:
        return "Amfibi"
    else:
        return "Darat"

df["habitat"] = df.apply(tentukan_habitat, axis=1)

def kelompok_kaki(legs):
    if legs == 0:
        return "Tanpa Kaki"
    elif legs == 2:
        return "Sedikit (2)"
    elif legs == 4:
        return "Sedang (4)"
    else:
        return "Banyak (6-8)"

df["leg_group"] = df["legs"].apply(kelompok_kaki)

df[["animal_name", "total_ciri_aktif", "habitat", "leg_group"]].head(10)


,animal_name,total_ciri_aktif,habitat,leg_group
0,aardvark,7,Darat,Sedang (4)
1,antelope,7,Darat,Sedang (4)
2,bass,7,Air,Tanpa Kaki
3,bear,7,Darat,Sedang (4)
4,boar,8,Darat,Sedang (4)
5,buffalo,7,Darat,Sedang (4)
6,calf,8,Darat,Sedang (4)
7,carp,7,Air,Tanpa Kaki
8,catfish,7,Air,Tanpa Kaki
9,cavy,6,Darat,Sedang (4)


Cek juga sebaran fitur kategorikal yang baru dibuat, biar tau distribusinya gimana:

In [10]:
print("Distribusi habitat:")
print(df["habitat"].value_counts())
print()
print("Distribusi leg_group:")
print(df["leg_group"].value_counts())


Distribusi habitat:
habitat
Darat     46
Air       31
Udara     19
Amfibi     5
Name: count, dtype: int64

Distribusi leg_group:
leg_group
Sedang (4)      38
Sedikit (2)     27
Tanpa Kaki      23
Banyak (6-8)    13
Name: count, dtype: int64


## 4. Encoding kolom kategorikal

Kolom object yang ada sekarang: `animal_name`, `class_name` (turunan target, nggak dipakai), `habitat`, sama `leg_group` (dua fitur baru).

- `animal_name` → ini cuma identifier, bukan fitur yang mewakili karakteristik hewan. Kalau di one-hot, hasilnya 101 kolom baru yang masing-masing cuma kepake di 1 baris — nggak guna buat modeling malah bikin overfitting. Jadi ini di-drop aja, bukan di-encode.
- `habitat` → nominal (Air/Udara/Darat/Amfibi nggak ada urutannya) → pakai One-Hot Encoding.
- `leg_group` → ordinal (ada urutan dari sedikit ke banyak) → pakai Ordinal Encoding, urutannya aku set manual.

In [11]:
from sklearn.preprocessing import OrdinalEncoder

df_encoded = pd.get_dummies(df, columns=["habitat"], prefix="habitat")

urutan_leg_group = ["Tanpa Kaki", "Sedikit (2)", "Sedang (4)", "Banyak (6-8)"]
ord_encoder = OrdinalEncoder(categories=[urutan_leg_group])
df_encoded["leg_group_encoded"] = ord_encoder.fit_transform(df_encoded[["leg_group"]])

df_encoded[["animal_name", "leg_group", "leg_group_encoded",
            "habitat_Air", "habitat_Udara", "habitat_Darat", "habitat_Amfibi"]].head(10)


,animal_name,leg_group,leg_group_encoded,habitat_Air,habitat_Udara,habitat_Darat,habitat_Amfibi
0,aardvark,Sedang (4),2.0,False,False,True,False
1,antelope,Sedang (4),2.0,False,False,True,False
2,bass,Tanpa Kaki,0.0,True,False,False,False
3,bear,Sedang (4),2.0,False,False,True,False
4,boar,Sedang (4),2.0,False,False,True,False
5,buffalo,Sedang (4),2.0,False,False,True,False
6,calf,Sedang (4),2.0,False,False,True,False
7,carp,Tanpa Kaki,0.0,True,False,False,False
8,catfish,Tanpa Kaki,0.0,True,False,False,False
9,cavy,Sedang (4),2.0,False,False,True,False


## 5. Susun dataset hasil preprocessing

Terakhir, susun dataset final: drop kolom yang udah nggak kepake lagi (`animal_name`, `legs` yang udah digantikan `legs_scaled`, `leg_group` yang udah digantikan `leg_group_encoded`, sama `class_name`). Kolom `class_type` tetep dipertahanin apa adanya sebagai label, nggak ikut dinormalisasi atau di-encode.

In [12]:
kolom_dibuang = ["animal_name", "legs", "leg_group", "class_name"]
df_final = df_encoded.drop(columns=kolom_dibuang)

print("Ukuran dataset akhir (baris, kolom):", df_final.shape)
print()
print("Kolom-kolom pada dataset hasil preprocessing:")
print(df_final.columns.tolist())
print()
df_final.head(10)


Ukuran dataset akhir (baris, kolom): (101, 23)

Kolom-kolom pada dataset hasil preprocessing:
['hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'tail', 'domestic', 'catsize', 'class_type', 'legs_scaled', 'total_ciri_aktif', 'habitat_Air', 'habitat_Amfibi', 'habitat_Darat', 'habitat_Udara', 'leg_group_encoded']



,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,...,domestic,catsize,class_type,legs_scaled,total_ciri_aktif,habitat_Air,habitat_Amfibi,habitat_Darat,habitat_Udara,leg_group_encoded
0,1,0,0,1,0,0,1,1,1,1,...,0,1,1,0.5,7,False,False,True,False,2.0
1,1,0,0,1,0,0,0,1,1,1,...,0,1,1,0.5,7,False,False,True,False,2.0
2,0,0,1,0,0,1,1,1,1,0,...,0,0,4,0.0,7,True,False,False,False,0.0
3,1,0,0,1,0,0,1,1,1,1,...,0,1,1,0.5,7,False,False,True,False,2.0
4,1,0,0,1,0,0,1,1,1,1,...,0,1,1,0.5,8,False,False,True,False,2.0
5,1,0,0,1,0,0,0,1,1,1,...,0,1,1,0.5,7,False,False,True,False,2.0
6,1,0,0,1,0,0,0,1,1,1,...,1,1,1,0.5,8,False,False,True,False,2.0
7,0,0,1,0,0,1,0,1,1,0,...,1,0,4,0.0,7,True,False,False,False,0.0
8,0,0,1,0,0,1,1,1,1,0,...,0,0,4,0.0,7,True,False,False,False,0.0
9,1,0,0,1,0,0,0,1,1,1,...,1,0,1,0.5,6,False,False,True,False,2.0


Dataset `df_final` ini udah bersih (nggak ada missing value, nama yang ambigu udah dibenerin), kolom `legs` udah dinormalisasi, kolom kategorikal udah di-encode sesuai jenisnya (one-hot buat nominal, ordinal buat yang ada urutan). Ada juga 3 fitur baru hasil feature engineering yang nggak nyentuh kolom target sama sekali. Dataset ini udah siap dipake buat modeling.

### Simpan hasil preprocessing (opsional)

hasilnya disimpan ke CSV baru biar bisa dipake lagi nanti tanpa harus ngulang semua proses dari awal.

In [13]:
df_final.to_csv("zoo_preprocessed.csv", index=False)
print("Dataset hasil preprocessing berhasil disimpan sebagai 'zoo_preprocessed.csv'")


Dataset hasil preprocessing berhasil disimpan sebagai 'zoo_preprocessed.csv'


## 6. Data Splitting

Tahap terakhir yaitu data splitting, bagi dataset jadi training data (buat latih model), validation data (buat tuning hyperparameter), sama testing data (buat uji akhir modelnya).

Pisahin dulu fitur (`X`) sama target (`y` = `class_type`), target nggak boleh ikut jadi fitur.

dibagi menjadi 3 bagian: 70% training, 15% validation, 15% testing, lewat 2 tahap `train_test_split()`:
1. Pisahin testing set (15%) dulu dari keseluruhan data.
2. Sisanya (85%) baru dipisah lagi jadi training sama validation.

Pakai `stratify=y` juga, soalnya jumlah data per kelas nggak seimbang (Mammalia 41, Amphibia cuma 4) — biar proporsi tiap kelas di training/validation/testing tetep mirip sama aslinya.

In [ ]:
from sklearn.model_selection import train_test_split


X = df_final.drop(columns=["class_type"])
y = df_final["class_type"]

print("Ukuran fitur X:", X.shape)
print("Ukuran target y:", y.shape)


Ukuran fitur X: (101, 22)
Ukuran target y: (101,)


In [15]:

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.15/0.85,
    random_state=42,
    stratify=y_train_val
)

print(f"Training set   : {X_train.shape[0]} baris ({X_train.shape[0]/len(df_final):.1%})")
print(f"Validation set  : {X_val.shape[0]} baris ({X_val.shape[0]/len(df_final):.1%})")
print(f"Testing set     : {X_test.shape[0]} baris ({X_test.shape[0]/len(df_final):.1%})")
print(f"Total           : {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} baris")


Training set   : 69 baris (68.3%)
Validation set  : 16 baris (15.8%)
Testing set     : 16 baris (15.8%)
Total           : 101 baris


Cek juga distribusi kelas (`class_type`) di tiap subset, buat mastiin stratify-nya kerja dengan bener.

In [16]:
import pandas as pd

perbandingan = pd.DataFrame({
    "Proporsi Asli": y.value_counts(normalize=True).sort_index(),
    "Training": y_train.value_counts(normalize=True).sort_index(),
    "Validation": y_val.value_counts(normalize=True).sort_index(),
    "Testing": y_test.value_counts(normalize=True).sort_index()
}).round(3)

perbandingan


,Proporsi Asli,Training,Validation,Testing
class_type,,,,
1,0.406,0.406,0.438,0.375
2,0.198,0.203,0.188,0.188
3,0.050,0.043,0.062,0.062
4,0.129,0.130,0.125,0.125
5,0.040,0.029,0.062,0.062
6,0.079,0.087,0.062,0.062
7,0.099,0.101,0.062,0.125


Distribusi kelasnya udah konsisten di ketiga subset, berarti pembagian datanya nggak condong ke kelas tertentu. `X_train`/`y_train`, `X_val`/`y_val`, sama `X_test`/`y_test` ini udah siap dipake buat training model di tugas berikutnya.